# KWS Negative sentences
Notebook for selecting and visualizing negative sentences (that contain no keyword).

In [11]:
from tira_kws.constants import (
    CAPSTONE_KEYWORDS,
    CAPSTONE_NEGATIVE_RECORDS,
    RECORD_LIST_CSV,
)

import pandas as pd
from unidecode import unidecode

In [6]:
kw_df = pd.read_csv(CAPSTONE_KEYWORDS)
kw_df

,keyword,keyword_id,gloss
0,àpɾí,0.0,boy
1,ðɔ̀mɔ̀cɔ̀,1.0,man
2,lɛ́ŋgɛ́n,2.0,mother
3,ŋɔ̀ɽíŋgɔ́,3.0,donkey
4,mùðù,4.0,leopard
5,ùɾnɔ̀,5.0,grandfather/grandchild
6,ðàŋàl,6.0,sheep
7,kə̀və̀lɛ̀ðɔ́,7.0,CLg-pull.PFV-VENT
8,kàŋú,8.0,CLg-see.PFV-VENT
9,íŋgá_nɔ̀nà,9.0,1sg-CLg-AUX_see.IPFV-IT


In [8]:
record_df = pd.read_csv(RECORD_LIST_CSV)
record_df.head()

,record_idx,recording_id,start,duration,channel,text,language,speaker,gloss,lemmata,translation,fst_normalized,unidecode_normalized
0,0,HH01082021,217.011,2.541,0,àprí jɜ̀dí ðáŋàlà,tic,Himidan,"boy[NOM,SG] touch[CLJ,VENT,PFV] sheep[ACC,SG,l...",àpɾí ɜd ðàŋàl,boy held sheep,àpɾí jɜ̀dí ðáŋàlà,apri jedi dangala
1,1,HH01082021,221.371,3.652,0,àprí jɜ̀dí ðáŋàlà,tic,Himidan,"boy[NOM,SG] touch[CLJ,VENT,PFV] sheep[ACC,SG,l...",àpɾí ɜd ðàŋàl,boy held sheep,àpɾí jɜ̀dí ðáŋàlà,apri jedi dangala
2,2,HH01082021,283.401,3.127,0,àprí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,tic,Himidan,"boy[NOM,SG] pull[CLJ,VENT,PFV] sheep[ACC,SG,le...",àpɾí vəlɛð ðàŋàl,boy pulled sheep (towards),àpɾí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,apri j@v@ledo dangala
3,3,HH01082021,288.835,2.745,0,àprí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,tic,Himidan,"boy[NOM,SG] pull[CLJ,VENT,PFV] sheep[ACC,SG,le...",àpɾí vəlɛð ðàŋàl,boy pulled sheep (towards),àpɾí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,apri j@v@ledo dangala
4,4,HH01082021,304.737,3.025,0,àprí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,tic,Himidan,"boy[NOM,SG] pull[CLJ,VENT,PFV] sheep[ACC,SG,le...",àpɾí vəlɛð ðàŋàl,boy pulled sheep (towards),àpɾí jə̀və̀lɛ̀ðɔ́ ðáŋàlà,apri j@v@ledo dangala


## Keyword mask
Select records that do not contain a unidecoded string for any of the 10 keywords.

In [12]:
keyword_mask = pd.Series([True]*len(record_df))

for keyword in kw_df['keyword'].unique():
    keyword_decode = unidecode(keyword)
    current_mask = ~record_df['unidecode_normalized'].str.contains(keyword_decode)
    keyword_mask &= current_mask

keyword_mask.sum()

np.int64(17824)

Now sample $n=100$ negative records and save them to CSV.

In [27]:
num_negative = 100
colmap = {
    "record_idx": "sentence_id",
    "translation": "translation",
    "text": "original_sentence",
    "fst_normalized": "textnorm_sentence",
}

negative_records = record_df[keyword_mask].sample(num_negative)
negative_records = negative_records.rename(columns=colmap)
negative_records = negative_records[colmap.values()]
negative_records.head()

,sentence_id,translation,original_sentence,textnorm_sentence
12856,12856,To feel good is good,ðícílá ðìcə̀lò,ðíbíl ðìcə̀lò
13375,13375,To cut meat is good,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò
6114,6114,they opened their mouths yesterday(and left),là kíðɛ̀ ŋɔ̀ɲà ùnɛ̀rɛ̀,là kíðɛ̀ ŋɔ̀ɲà ùnɛ́ɾɛ́
18219,18219,a co-wife is good,ɛ̀ɾə̀mt̪ù kìcəlò,rə̀mt̪ kìcə̀lò
12417,12417,Don't help Kuku!,àt̪ ɔ́ɽɟɛ̀cɔ̀ŋnɔ̀ kúkùŋú ánó dɛ̀,át̪ú ɔ́ɽɟɛ̀cìnɔ̀ kúkùŋú ánó dɛ̀


Load `keyword_negatives.csv` and merge data (overwrite existing rows but keep column names).

In [31]:
neg_df = pd.read_csv(CAPSTONE_NEGATIVE_RECORDS)

neg_df = neg_df.drop(neg_df.index)
neg_df = pd.concat([neg_df, negative_records])

neg_df.head()

,sentence_id,translation,original_sentence,textnorm_sentence,audionorm_sentence,audio_quality,extra_speech,missing_speech,mistranscription,comment
12856,12856,To feel good is good,ðícílá ðìcə̀lò,ðíbíl ðìcə̀lò,NaN,NaN,NaN,NaN,NaN,NaN
13375,13375,To cut meat is good,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,ðə́ðá ɛ́ðɛ̀ ánó ðìcə̀lò,NaN,NaN,NaN,NaN,NaN,NaN
6114,6114,they opened their mouths yesterday(and left),là kíðɛ̀ ŋɔ̀ɲà ùnɛ̀rɛ̀,là kíðɛ̀ ŋɔ̀ɲà ùnɛ́ɾɛ́,NaN,NaN,NaN,NaN,NaN,NaN
18219,18219,a co-wife is good,ɛ̀ɾə̀mt̪ù kìcəlò,rə̀mt̪ kìcə̀lò,NaN,NaN,NaN,NaN,NaN,NaN
12417,12417,Don't help Kuku!,àt̪ ɔ́ɽɟɛ̀cɔ̀ŋnɔ̀ kúkùŋú ánó dɛ̀,át̪ú ɔ́ɽɟɛ̀cìnɔ̀ kúkùŋú ánó dɛ̀,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
neg_df.to_csv(CAPSTONE_NEGATIVE_RECORDS)